In [ ]:
import numpy as np

x = np.array([2.0, 4.0, 6.0, 8.0])

In [ ]:
mean = np.mean(x)
var = np.mean((x - mean) ** 2)
std = np.sqrt(var)

print("mean:", mean)
print("var:", var)
print("std:", std)

mean: 5.0
var: 5.0
std: 2.23606797749979


In [ ]:
normalized = (x - mean) / std

print("normalized:", normalized)
print("new mean:", np.mean(normalized))
print("new std:", np.std(normalized))

normalized: [-1.34164079 -0.4472136   0.4472136   1.34164079]
new mean: 0.0
new std: 1.0


In [ ]:
def layer_norm(x, eps=1e-5):
    mean = np.mean(x)
    var = np.mean((x - mean) ** 2)
    return (x - mean) / np.sqrt(var + eps)

print(layer_norm(np.array([2.0, 4.0, 6.0, 8.0])))
print(layer_norm(np.array([10.0, 30.0, 50.0, 70.0])))

[-1.34163944 -0.44721315  0.44721315  1.34163944]
[-1.34164077 -0.44721359  0.44721359  1.34164077]


In [ ]:
import torch
import torch.nn as nn

ln = nn.LayerNorm(4)
t = torch.tensor([2.0, 4.0, 6.0, 8.0])
print(ln(t))

tensor([-1.3416, -0.4472,  0.4472,  1.3416],
       grad_fn=<NativeLayerNormBackward0>)


In [ ]:
import torch
import torch.nn as nn

vocab_size = 10    # number of different tokens
block_size = 6     # longest sentence allowed
n_embd = 4         # size of each vector

tok_emb = nn.Embedding(vocab_size, n_embd)
pos_emb = nn.Embedding(block_size, n_embd)

ids = torch.tensor([3, 1, 4, 1, 5])   # a 5-token "sentence"

tokens = tok_emb(ids)                  # what each word is
positions = torch.arange(len(ids))     # [0, 1, 2, 3, 4]
pos = pos_emb(positions)               # where each word is
x = tokens + pos

print(tokens.shape, pos.shape, x.shape)
print("tokens row 1 vs row 3:")
print(tokens[1], tokens[3])
print("x row 1 vs row 3:")
print(x[1], x[3])

torch.Size([5, 4]) torch.Size([5, 4]) torch.Size([5, 4])
tokens row 1 vs row 3:
tensor([-0.0185,  0.9040, -0.0864,  0.5271], grad_fn=<SelectBackward0>) tensor([-0.0185,  0.9040, -0.0864,  0.5271], grad_fn=<SelectBackward0>)
x row 1 vs row 3:
tensor([0.1325, 1.2835, 0.0212, 1.0517], grad_fn=<SelectBackward0>) tensor([-0.5114,  1.0039,  1.3323,  1.4676], grad_fn=<SelectBackward0>)


In [ ]:
import torch.nn.functional as F

head_size = 4
key = nn.Linear(n_embd, head_size, bias=False)
query = nn.Linear(n_embd, head_size, bias=False)
value = nn.Linear(n_embd, head_size, bias=False)

xn = nn.LayerNorm(n_embd)(x)            # Norm comes first, as in the diagram

k = key(xn)
q = query(xn)
v = value(xn)

scores = q @ k.T / head_size ** 0.5     # how much each word looks at each other word
T = xn.shape[0]
mask = torch.tril(torch.ones(T, T))     # a word may only look at earlier words
scores = scores.masked_fill(mask == 0, float('-inf'))
weights = F.softmax(scores, dim=-1)     # each row adds up to 1
out = weights @ v                       # each word gathers information from earlier words

print(weights)
print(out.shape)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5555, 0.4445, 0.0000, 0.0000, 0.0000],
        [0.3302, 0.2992, 0.3706, 0.0000, 0.0000],
        [0.2415, 0.2412, 0.2328, 0.2845, 0.0000],
        [0.2334, 0.1970, 0.2254, 0.1367, 0.2075]], grad_fn=<SoftmaxBackward0>)
torch.Size([5, 4])


In [ ]:
# the "+" after attention (residual connection)
x2 = x + out

# second Norm, then feed-forward
x2n = nn.LayerNorm(n_embd)(x2)
ffwd = nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),   # expand
    nn.ReLU(),
    nn.Linear(4 * n_embd, n_embd),   # shrink back
)
f = ffwd(x2n)

# the "+" after feed-forward (residual connection)
x3 = x2 + f

print(x.shape, x2.shape, x3.shape)
print(x3)

torch.Size([5, 4]) torch.Size([5, 4]) torch.Size([5, 4])
tensor([[-0.9873, -0.6083, -1.5770,  0.9580],
        [-0.7638,  0.8009, -0.4540,  1.1866],
        [ 0.7472, -1.0313, -0.6558, -0.4041],
        [-0.8640,  0.7423,  1.2918,  1.5249],
        [-0.3864,  0.9841, -1.2525, -0.0226]], grad_fn=<AddBackward0>)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MyLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))   # learnable scale
        self.shift = nn.Parameter(torch.zeros(dim))  # learnable shift

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, unbiased=False, keepdim=True)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


class Block(nn.Module):
    def __init__(self, n_embd, block_size, use_ln=True):
        super().__init__()
        self.key = nn.Linear(n_embd, n_embd, bias=False)
        self.query = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd, n_embd, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd)
        )
        self.ln1 = MyLayerNorm(n_embd) if use_ln else nn.Identity()
        self.ln2 = MyLayerNorm(n_embd) if use_ln else nn.Identity()

    def forward(self, x):
        h = self.ln1(x)                                   # Norm
        B, T, C = h.shape
        k, q, v = self.key(h), self.query(h), self.value(h)
        scores = q @ k.transpose(-2, -1) / C ** 0.5       # Attention
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        w = F.softmax(scores, dim=-1)
        x = x + w @ v                                     # + (residual)
        x = x + self.ffwd(self.ln2(x))                    # Norm, Feed-forward, + (residual)
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=32, n_layer=2, use_ln=True):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)   # Token embedding
        self.pos_emb = nn.Embedding(block_size, n_embd)   # Position embedding
        self.blocks = nn.Sequential(*[Block(n_embd, block_size, use_ln) for _ in range(n_layer)])
        self.ln_f = MyLayerNorm(n_embd) if use_ln else nn.Identity()
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        x = self.tok_emb(ids) + self.pos_emb(torch.arange(T))
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


# quick tests
t = torch.tensor([[2.0, 4.0, 6.0, 8.0]])
print(MyLayerNorm(4)(t))                                  # should match nn.LayerNorm(4)(t)

model = TinyGPT(vocab_size=10, block_size=6)
logits, loss = model(torch.tensor([[3, 1, 4, 1, 5]]))
print(logits.shape)                                       # expect [1, 5, 10]

tensor([[-1.3416, -0.4472,  0.4472,  1.3416]], grad_fn=<AddBackward0>)
torch.Size([1, 5, 10])


In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("DigitalUmuganda/kinyarwanda-english-machine-translation-dataset", repo_type="dataset")
print(files)

['.gitattributes', 'README.md', 'kinyarwanda-english-corpus.tsv', 'kinyarwanda-english-corpus2.tsv', 'kinyarwanda-english-corpus3.tsv']


In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="DigitalUmuganda/kinyarwanda-english-machine-translation-dataset",
    filename="kinyarwanda-english-corpus.tsv",
    repo_type="dataset",
)

try:
    df = pd.read_csv(path, sep="\t", quoting=3, on_bad_lines="skip",
                     header=None, names=["rw", "en"])
except UnicodeDecodeError:
    df = pd.read_csv(path, sep="\t", quoting=3, on_bad_lines="skip",
                     header=None, names=["rw", "en"], encoding="cp1252")

df = df.dropna()
for col in ["rw", "en"]:
    df[col] = df[col].str.strip().str.lower().str.replace("’", "'", regex=False)

df = df[(df["rw"].str.len() <= 40) & (df["en"].str.len() <= 40)]
df = df.sample(n=min(3000, len(df)), random_state=0).reset_index(drop=True)

# fix rows where the two languages are swapped
en_words = {"the","a","an","is","are","was","were","to","of","and","in","that","this",
            "he","she","it","you","i","we","they","his","her","my","for","with","not",
            "have","had","be","what","who","at","on","your","our","their"}

def en_score(s):
    w = s.split()
    return sum(x in en_words for x in w) / max(len(w), 1)

swap = df["rw"].apply(en_score) > df["en"].apply(en_score)
print("rows that looked swapped:", swap.sum())

df.loc[swap, ["rw", "en"]] = df.loc[swap, ["en", "rw"]].values

print(df.columns.tolist())
print(df.shape)
print(df.head(10))

kinyarwanda-english-corpus.tsv:   0%|          | 0.00/1.17M [00:00<?, ?B/s]

rows that looked swapped: 11
['rw', 'en']
(3000, 2)
                                     rw                                   en
0                         gutumutwe kwe                     loosing her mind
1                 ntushobora kubiryozwa                   you can't blame it
2                      ibi bibaho cyane                   this happens a lot
3                        nibyo ariburye             that is what he will eat
4        kuko nawe yagombaga gusimburwa   because he also had to be replaced
5        iki cyuma ntabwo gityaye cyane         this metal is not very sharp
6  sinshobora kwizera ko wankorera ibyo   i can't believe you did that to me
7         dore agasanduku k'ibitekerezo           look at the suggestion box
8             nzakurimburira mu mahanga  i will destroy you in foreign lands
9          ndisukira ikirahure cy'amazi  i will pour myself a glass of water


In [ ]:
lines = [f"{r} = {e}\n" for r, e in zip(df["rw"], df["en"])]
text = "".join(lines)

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

print("vocab size:", vocab_size)
print("characters in text:", len(data))
print(text[:150])

vocab size: 49
characters in text: 174539
gutumutwe kwe = loosing her mind
ntushobora kubiryozwa = you can't blame it
ibi bibaho cyane = this happens a lot
nibyo ariburye = that is what he wil


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

block_size = 96
batch_size = 32

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def val_loss(model, iters=20):
    model.eval()
    losses = [model(*get_batch("val"))[1].item() for _ in range(iters)]
    model.train()
    return sum(losses) / len(losses)

def train(use_ln, steps=1500, lr=3e-3):
    torch.manual_seed(0)
    model = TinyGPT(vocab_size, block_size, n_embd=64, n_layer=2, use_ln=use_ln).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    history = []
    for step in range(steps + 1):
        if step % 250 == 0:
            vl = val_loss(model)
            history.append((step, vl))
            print(f"use_ln={use_ln} step {step}: val loss {vl:.3f}")
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model, history

model_ln, hist_ln = train(use_ln=True)
model_no, hist_no = train(use_ln=False)

use_ln=True step 0: val loss 4.010
use_ln=True step 250: val loss 2.170
use_ln=True step 500: val loss 1.920
use_ln=True step 750: val loss 1.792
use_ln=True step 1000: val loss 1.733
use_ln=True step 1250: val loss 1.672
use_ln=True step 1500: val loss 1.632
use_ln=False step 0: val loss 4.206
use_ln=False step 250: val loss 2.108
use_ln=False step 500: val loss 1.887
use_ln=False step 750: val loss 1.771
use_ln=False step 1000: val loss 1.725
use_ln=False step 1250: val loss 1.684
use_ln=False step 1500: val loss 1.652


In [ ]:
def train_stress(use_ln, n_layer=6, lr=1e-2, steps=400):
    torch.manual_seed(0)
    model = TinyGPT(vocab_size, block_size, n_embd=64, n_layer=n_layer, use_ln=use_ln).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps + 1):
        if step % 100 == 0:
            print(f"use_ln={use_ln} step {step}: val loss {val_loss(model):.3f}")
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model

m_ln = train_stress(use_ln=True)
m_no = train_stress(use_ln=False)

use_ln=True step 0: val loss 4.049
use_ln=True step 100: val loss 2.351
use_ln=True step 200: val loss 2.228
use_ln=True step 300: val loss 2.119
use_ln=True step 400: val loss 1.996
use_ln=False step 0: val loss 4.749
use_ln=False step 100: val loss 2.453
use_ln=False step 200: val loss 2.320
use_ln=False step 300: val loss 2.203
use_ln=False step 400: val loss 2.106


In [ ]:
def block_stats(model):
    model.eval()
    xb, _ = get_batch("val")
    with torch.no_grad():
        x = model.tok_emb(xb) + model.pos_emb(torch.arange(xb.shape[1], device=xb.device))
        stats = [x.std().item()]            # size of the numbers at the start
        for blk in model.blocks:
            x = blk(x)
            stats.append(x.std().item())    # size after each block
    return stats

print("with LN   :", [round(s, 2) for s in block_stats(m_ln)])
print("without LN:", [round(s, 2) for s in block_stats(m_no)])

with LN   : [1.45, 9.92, 10.77, 11.35, 12.4, 14.02, 15.77]
without LN: [1.27, 1.36, 1.34, 1.5, 1.81, 2.03, 2.86]


In [ ]:
def sublayer_input_stats(model):
    model.eval()
    xb, _ = get_batch("val")
    attn_in, ffn_in, hooks = [], [], []
    for blk in model.blocks:
        hooks.append(blk.key.register_forward_hook(lambda m, i, o: attn_in.append(i[0].std().item())))
        hooks.append(blk.ffwd.register_forward_hook(lambda m, i, o: ffn_in.append(i[0].std().item())))
    with torch.no_grad():
        model(xb)
    for h in hooks:
        h.remove()
    return attn_in, ffn_in

for name, m in [("with LN", m_ln), ("without LN", m_no)]:
    a, f = sublayer_input_stats(m)
    print(name, "attention input:", [round(s, 2) for s in a])
    print(name, "feed-forward input:", [round(s, 2) for s in f])

with LN attention input: [0.65, 0.75, 0.83, 0.86, 0.9, 0.82]
with LN feed-forward input: [1.12, 0.82, 0.81, 0.93, 1.02, 1.04]
without LN attention input: [1.27, 1.35, 1.33, 1.49, 1.79, 2.01]
without LN feed-forward input: [1.28, 1.4, 1.41, 1.6, 1.94, 2.13]


In [ ]:
@torch.no_grad()
def translate(model, rw_sentence, max_new=60):
    model.eval()
    prompt = rw_sentence.lower().strip().replace("’", "'") + " = "
    ids = [stoi[c] for c in prompt if c in stoi]
    x = torch.tensor([ids], device=device)
    for _ in range(max_new):
        logits, _ = model(x[:, -block_size:])
        nxt = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)   # greedy choice
        x = torch.cat([x, nxt], dim=1)
        if itos[nxt.item()] == "\n":
            break
    return decode(x[0].tolist()).split(" = ", 1)[-1].strip()

for i in range(len(df) - 5, len(df)):
    rw, en = df["rw"][i], df["en"][i]
    print("Kinyarwanda :", rw)
    print("true English:", en)
    print("with LN     :", translate(model_ln, rw))
    print("without LN  :", translate(model_no, rw))
    print()

Kinyarwanda : uru rugi numukara
true English: this door is black
with LN     : this is the was to see this
without LN  : i want this the was to say

Kinyarwanda : yamanitse mbere yuko ngira icyo mvuga.
true English: he hung up before i could say anything.
with LN     : there is a should the way
without LN  : i want to do this that the time this a come the this me simo

Kinyarwanda : narinzi ko utazabikora.
true English: i thought you wouldn't do it.
with LN     : i don't this will this way this this
without LN  : i don't know this the the people come the the time this

Kinyarwanda : sinzi niba agenda cyangwa ahaguma.
true English: i don't know if he is going or staying.
with LN     : this is not is the was the way
without LN  : this is not this a come this the time the should this is not

Kinyarwanda : reka ngufashe muri abi.
true English: let me help you with this
with LN     : this is not the was the was to see the was the was to seen
without LN  : i don't know this the the people co

In [ ]:
for name in ["model_ln", "model_no", "chars", "vocab_size", "block_size", "df"]:
    print(name, "OK" if name in globals() else "MISSING")

model_ln OK
model_no OK
chars OK
vocab_size OK
block_size OK
df OK


In [ ]:
import json
from google.colab import files

torch.save(model_ln.state_dict(), "model_ln.pt")
torch.save(model_no.state_dict(), "model_no.pt")

cfg = {
    "chars": chars,
    "vocab_size": vocab_size,
    "block_size": block_size,
    "n_embd": 64,
    "n_layer": 2,
    "examples": df["rw"].head(5).tolist(),
}
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False)

for name in ["model_ln.pt", "model_no.pt", "config.json"]:
    files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>